In [14]:
import pandas as pd

df = pd.read_csv('../data/tmdb_movies_ru.csv')

df.shape

(18092, 8)

In [15]:
df.head()

,id,title,genres,overview,poster_path,popularity,vote_average,trailer_url
0,1368337,Одиссея,"['приключения', 'боевик', 'фэнтези']","После Троянской войны Одиссей, царь Итаки, во ...",/n7NR7SH7CyiiF70yN96r3d1jWr0.jpg,1049.7104,8.000,https://www.youtube.com/watch?v=xjLLXdYldzA
1,1275779,День разоблачения,"['фантастика', 'триллер']",Человечество узнаёт о прибытии на Землю инопла...,/rwsWNNJKqLdN0ymCPx1yJagMVi.jpg,596.4318,7.378,https://www.youtube.com/watch?v=ghzGX-FPzhM
2,454639,Властелины вселенной,"['боевик', 'фэнтези', 'фантастика']","Адам, 10-летний принц с планеты Этерния, терпи...",/u0SxcVBFf9c5PFt9i12RYSLoQf.jpg,539.6908,7.342,NaN
3,1108427,Моана,"['семейный', 'фэнтези', 'комедия', 'приключения']",Моана откликается на зов океана. Впервые в жиз...,/s7o0gbvgpjwb8S0Uc5QaEdwJntb.jpg,519.3627,5.900,https://www.youtube.com/watch?v=EE7oLAiCfNU
4,1339713,Обсессия,"['ужасы', 'триллер']",Безнадёжный романтик Беар давно и безответно в...,/cdRHtfzxbpjxD5yOyFrRummb0yO.jpg,401.1034,8.254,https://www.youtube.com/watch?v=NDJGqOOf-wM


In [16]:
df.isna().sum()

id                  0
title               0
genres              0
overview            0
poster_path       205
popularity          0
vote_average        0
trailer_url     12822
dtype: int64

In [17]:
df = df.dropna(subset=['poster_path'])

df = df[df['poster_path'].astype(str).str.strip() != '']

df

,id,title,genres,overview,poster_path,popularity,vote_average,trailer_url
0,1368337,Одиссея,"['приключения', 'боевик', 'фэнтези']","После Троянской войны Одиссей, царь Итаки, во ...",/n7NR7SH7CyiiF70yN96r3d1jWr0.jpg,1049.7104,8.000,https://www.youtube.com/watch?v=xjLLXdYldzA
1,1275779,День разоблачения,"['фантастика', 'триллер']",Человечество узнаёт о прибытии на Землю инопла...,/rwsWNNJKqLdN0ymCPx1yJagMVi.jpg,596.4318,7.378,https://www.youtube.com/watch?v=ghzGX-FPzhM
2,454639,Властелины вселенной,"['боевик', 'фэнтези', 'фантастика']","Адам, 10-летний принц с планеты Этерния, терпи...",/u0SxcVBFf9c5PFt9i12RYSLoQf.jpg,539.6908,7.342,NaN
3,1108427,Моана,"['семейный', 'фэнтези', 'комедия', 'приключения']",Моана откликается на зов океана. Впервые в жиз...,/s7o0gbvgpjwb8S0Uc5QaEdwJntb.jpg,519.3627,5.900,https://www.youtube.com/watch?v=EE7oLAiCfNU
4,1339713,Обсессия,"['ужасы', 'триллер']",Безнадёжный романтик Беар давно и безответно в...,/cdRHtfzxbpjxD5yOyFrRummb0yO.jpg,401.1034,8.254,https://www.youtube.com/watch?v=NDJGqOOf-wM
...,...,...,...,...,...,...,...,...
18087,617499,Черкассы,"['военный', 'боевик', 'драма']",Февраль 2014. Начинается оккупация Крымского п...,/2kiTRYgjs2Nna7HTmasdUGtFFQX.jpg,1.8317,6.464,NaN
18088,53487,Где бы ты ни был,['драма'],"Рок-певец, скучающий на пенсии, отправляется н...",/k7ljIVOcgRnW3CIpSNx2DqgRMYW.jpg,1.8316,6.973,NaN
18089,1537304,А жизнь продолжается,['драма'],Молодой киевлянин Семен Дахно тихо и скромно ж...,/j7gEdiVq8MduTGYHVKALua2mI7K.jpg,1.8316,0.000,NaN
18090,532030,Призрак секса,['комедия'],"Три девушки, Эмбер, Бренди и Кэсси, сталкивают...",/fPEgpDm5K7uFg9v84wE5nNmDZLW.jpg,1.8315,5.241,NaN


In [18]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('cointegrated/rubert-tiny2')

texts_to_embed = []
for _, row in df.iterrows():
    title = str(row['title']) if pd.notna(row['title']) else ""
    genres = str(row['genres']) if pd.notna(row['genres']) else ""
    overview = str(row['overview']) if pd.notna(row['overview']) else ""
    
    combined_text = f"Название: {title}. Жанры: {genres}. Описание: {overview}"
    texts_to_embed.append(combined_text)

embeddings = model.encode(
    texts_to_embed, 
    batch_size=64, 
    show_progress_bar=True, 
    normalize_embeddings=True
)

print(f"Успешно сгенерировано {len(embeddings)} векторов.")
print(f"Размерность одного вектора: {embeddings.shape[1]}")

bin_path = '../data/movie_embeddings.bin'

with open(bin_path, 'wb') as f:
    np.array(embeddings.shape, dtype=np.uint32).tofile(f)
    embeddings.astype(np.float32).tofile(f)

df[['id', 'title']].to_csv('../data/movie_mapping.csv', index=False)
print("Векторы и маппинг сохранены!")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/280 [00:00<?, ?it/s]

Успешно сгенерировано 17887 векторов.
Размерность одного вектора: 312
Векторы и маппинг сохранены!


In [20]:
df = pd.read_csv('../data/movie_mapping.csv')

movie_ids = df['id'].values.astype(np.uint32)
movie_ids.tofile('../data/movie_ids.bin')

print(f"Сохранено {len(movie_ids)} ID!")

Сохранено 17887 ID!
